# Selection of the number of PLS components

Supplementary figure supporting the choice of **10 PLS latent components** for all
semantic-regression analyses.

We sweep `n_components` from **2 to 35** for the main **kernel-PLS** model
(Nystroem-RBF + PLSRegression) with **GloVe** targets, across all **12 participants**,
evaluating peak-across-time-bin held-out performance over repeated train/test splits
(10 splits per setting). We report, as a function of component count:

1. **Balanced (category-independent) category accuracy** — held-out 1-NN cosine retrieval
2. **Balanced word accuracy** — held-out 1-NN cosine retrieval
3. **Train vs. test cosine similarity** — and the growing train−test generalization gap

Source data: `main/tests/results/pls_lc_{PATIENT}.csv` (produced by
`tests/model_diagnostics/pls_components_sweep.py`).

All PDFs are written with `pdf.fonttype = 42` (TrueType) so text stays editable in
Illustrator. Figures → this folder; plotted CSVs → `source_data/`.


In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

# Editable-text PDFs/PS (TrueType, not Type-3 outlines)
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['svg.fonttype'] = 'none'
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.size'] = 9
mpl.rcParams['axes.spines.top'] = False
mpl.rcParams['axes.spines.right'] = False

MAIN_DIR = Path(r"d:/OneDrive - Northwestern University/PycharmProjects/Speech/main")
RESULTS  = MAIN_DIR / 'tests' / 'results'
FIG_DIR  = MAIN_DIR / 'figures_for_paper' / 'pls_components'
SRC_DIR  = FIG_DIR / 'source_data'
SRC_DIR.mkdir(parents=True, exist_ok=True)

EMB   = 'GloVe'
MODEL = 'kernel_pls'
NMAX  = 35
CHOSEN_N = 10
print('FIG_DIR:', FIG_DIR)


## Load & filter the sweep source data

In [ ]:
frames = []
for f in sorted(RESULTS.glob('pls_lc_*.csv')):
    frames.append(pd.read_csv(f))
raw = pd.concat(frames, ignore_index=True)

df = raw[(raw.embedding == EMB) & (raw.model == MODEL) & (raw.n_components <= NMAX)].copy()
df = df.drop_duplicates(subset=['patient', 'embedding', 'model', 'n_components', 'epoch'])

patients = sorted(df.patient.unique())
comps    = sorted(df.n_components.unique())
print(f'{len(patients)} patients:', patients)
print('n_components:', comps)
print('epochs/setting:', df.groupby(["patient","n_components"]).size().unique())
assert len(patients) == 12, f'expected 12 patients, found {len(patients)}'


## Aggregate

Per participant we average across the repeated train/test splits, then take the
mean ± SEM **across participants** (n = 12). This hierarchical aggregation keeps each
participant weighted equally regardless of trial count.

In [ ]:
METRICS = {
    'cat_bal_acc':  'Balanced category accuracy',
    'word_bal_acc': 'Balanced word accuracy',
    'test_cosine':  'Held-out cosine similarity',
    'train_cosine': 'Train cosine similarity',
}

# 1) per-patient mean over epochs
per_pat = (df.groupby(['patient', 'n_components'])[list(METRICS)]
             .mean().reset_index())

# 2) mean +/- SEM across patients
def agg_across(col):
    g = per_pat.groupby('n_components')[col]
    m = g.mean()
    sem = g.std(ddof=1) / np.sqrt(g.count())
    return m, sem

grand = pd.DataFrame({'n_components': comps}).set_index('n_components')
for col in METRICS:
    m, sem = agg_across(col)
    grand[col + '_mean'] = m
    grand[col + '_sem']  = sem
grand['cosine_gap_mean'] = grand['train_cosine_mean'] - grand['test_cosine_mean']
# gap SEM from per-patient gaps
per_pat['cosine_gap'] = per_pat['train_cosine'] - per_pat['test_cosine']
gg = per_pat.groupby('n_components')['cosine_gap']
grand['cosine_gap_sem'] = gg.std(ddof=1) / np.sqrt(gg.count())
grand = grand.reset_index()

grand.round(4)


In [ ]:
# Save source data (grand mean + per-patient tidy table)
grand.to_csv(SRC_DIR / 'pls_components_grandmean.csv', index=False)
per_pat.to_csv(SRC_DIR / 'pls_components_per_patient.csv', index=False)

# Quick text of the values quoted in the manuscript
def at(col, n):
    return float(grand.loc[grand.n_components == n, col].iloc[0])
print(f"category accuracy @ n={CHOSEN_N}: {at('cat_bal_acc_mean', CHOSEN_N):.3f}")
for n in [2, 10, 15, 20]:
    if n in comps:
        print(f"train-test cosine gap @ n={n}: {at('cosine_gap_mean', n):.3f}")


## Plot helpers

In [ ]:
PAT_COLOR = '#b8c4d9'   # thin individual-patient lines
MEAN_COLOR = '#1f4e8c'

def plot_metric(ax, col, ylabel, show_patients=True, color=MEAN_COLOR):
    x = grand['n_components'].to_numpy()
    m = grand[col + '_mean'].to_numpy()
    s = grand[col + '_sem'].to_numpy()
    if show_patients:
        for pat, g in per_pat.groupby('patient'):
            g = g.sort_values('n_components')
            ax.plot(g['n_components'].to_numpy(), g[col].to_numpy(), color=PAT_COLOR,
                    lw=0.8, alpha=0.7, zorder=1)
    ax.fill_between(x, m - s, m + s, color=color, alpha=0.20, lw=0, zorder=2)
    ax.plot(x, m, color=color, lw=2.2, marker='o', ms=4, zorder=3)
    ax.axvline(CHOSEN_N, color='#888', ls='--', lw=1, zorder=0)
    ax.set_xlabel('Number of PLS components')
    ax.set_ylabel(ylabel)
    ax.set_xticks([2, 6, 10, 15, 20, 25, 30, 35])
    return ax


## Figure 1 — Balanced category accuracy

In [ ]:
fig, ax = plt.subplots(figsize=(3.4, 2.8))
plot_metric(ax, 'cat_bal_acc', 'Balanced category accuracy', color='#c0392b')
ax.annotate(f"n={CHOSEN_N}", xy=(CHOSEN_N, ax.get_ylim()[0]),
            xytext=(CHOSEN_N + 0.6, ax.get_ylim()[0] + 0.01*(ax.get_ylim()[1]-ax.get_ylim()[0])),
            fontsize=8, color='#555')
fig.tight_layout()
fig.savefig(FIG_DIR / '01_category_accuracy_vs_components.pdf', bbox_inches='tight')
fig.savefig(FIG_DIR / '01_category_accuracy_vs_components.png', dpi=200, bbox_inches='tight')
plt.show()


## Figure 2 — Balanced word accuracy

In [ ]:
fig, ax = plt.subplots(figsize=(3.4, 2.8))
plot_metric(ax, 'word_bal_acc', 'Balanced word accuracy', color='#27ae60')
fig.tight_layout()
fig.savefig(FIG_DIR / '02_word_accuracy_vs_components.pdf', bbox_inches='tight')
fig.savefig(FIG_DIR / '02_word_accuracy_vs_components.png', dpi=200, bbox_inches='tight')
plt.show()


## Figure 3 — Train vs. test cosine similarity (generalization gap)

Held-out (test) cosine plateaus while train cosine keeps rising: the shaded region is
the train−test gap, a direct overfitting diagnostic that grows monotonically with
component count.

In [ ]:
fig, ax = plt.subplots(figsize=(3.8, 2.8))
x = grand['n_components'].to_numpy()
tm, ts = grand['test_cosine_mean'].to_numpy(),  grand['test_cosine_sem'].to_numpy()
rm, rs = grand['train_cosine_mean'].to_numpy(), grand['train_cosine_sem'].to_numpy()

ax.fill_between(x, tm, rm, color='#f0a500', alpha=0.15, lw=0, label='train−test gap')
ax.fill_between(x, rm - rs, rm + rs, color='#888', alpha=0.15, lw=0)
ax.plot(x, rm, color='#555', lw=2, ls='--', marker='s', ms=3.5, label='train')
ax.fill_between(x, tm - ts, tm + ts, color='#1f4e8c', alpha=0.2, lw=0)
ax.plot(x, tm, color='#1f4e8c', lw=2.2, marker='o', ms=4, label='held-out (test)')
ax.axvline(CHOSEN_N, color='#888', ls=':', lw=1)
ax.set_xlabel('Number of PLS components')
ax.set_ylabel('Cosine similarity')
ax.set_xticks([2, 6, 10, 15, 20, 25, 30, 35])
ax.legend(frameon=False, fontsize=7.5, loc='center right')
fig.tight_layout()
fig.savefig(FIG_DIR / '03_traintest_cosine_vs_components.pdf', bbox_inches='tight')
fig.savefig(FIG_DIR / '03_traintest_cosine_vs_components.png', dpi=200, bbox_inches='tight')
plt.show()


## Combined 3-panel supplementary figure

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(9.6, 2.9))
plot_metric(axes[0], 'cat_bal_acc', 'Balanced category accuracy', color='#c0392b')
plot_metric(axes[1], 'word_bal_acc', 'Balanced word accuracy', color='#27ae60')

ax = axes[2]
ax.fill_between(x, tm, rm, color='#f0a500', alpha=0.15, lw=0, label='train−test gap')
ax.plot(x, rm, color='#555', lw=2, ls='--', marker='s', ms=3.5, label='train')
ax.fill_between(x, tm - ts, tm + ts, color='#1f4e8c', alpha=0.2, lw=0)
ax.plot(x, tm, color='#1f4e8c', lw=2.2, marker='o', ms=4, label='held-out')
ax.axvline(CHOSEN_N, color='#888', ls=':', lw=1)
ax.set_xlabel('Number of PLS components'); ax.set_ylabel('Cosine similarity')
ax.set_xticks([2, 6, 10, 15, 20, 25, 30, 35])
ax.legend(frameon=False, fontsize=7, loc='center right')

for lab, a in zip('abc', axes):
    a.set_title(lab, loc='left', fontweight='bold', fontsize=11)
fig.tight_layout()
fig.savefig(FIG_DIR / '00_pls_components_selection_combined.pdf', bbox_inches='tight')
fig.savefig(FIG_DIR / '00_pls_components_selection_combined.png', dpi=200, bbox_inches='tight')
plt.show()
print('Saved figures to', FIG_DIR)
